# Datathon 2026 – Task 1: Traffic Speed Forecasting (LightGBM v2)
Improvements over v1:
- **2.5x more training data** (STEP=2, ~10M rows)
- **1000 trees** + lower learning rate 0.03 + early stopping 100
- **num_leaves=255** (more expressive trees)
- **4 rolling features**: mean_5, std_15, slope, neigh_mean_5
- **Vectorized data building** (numpy only, no Python loop)


## Cara pakai

**Kaggle** — Attach dataset Task 1 via *Add Data*, langsung *Run All*.

**Google Colab** — Upload `datathon-task-1.zip` ke sidebar file, lalu *Run All*.
Kalau gagal, isi `DATA_PATH` di cell pertama dengan path yang benar.


In [1]:
import numpy as np
import pandas as pd
import json, os, time, zipfile
from pathlib import Path
import lightgbm as lgb
from sklearn.model_selection import train_test_split

# ══════════════════════════════════════════════════════════════════════════════
# MANUAL OVERRIDE — isi path di sini kalau auto-detection gagal
DATA_PATH = ''
# ══════════════════════════════════════════════════════════════════════════════

def find_data_root() -> Path:
    if DATA_PATH:
        p = Path(DATA_PATH)
        if (p / 'train').exists(): return p
    for zip_name in ['datathon-task-1.zip', 'datathon-task1.zip', 'dataset-task1.zip']:
        z = Path(zip_name)
        if z.exists():
            print(f'Extracting {zip_name}...')
            with zipfile.ZipFile(z) as zf: zf.extractall('.')
            break
    kaggle_in = Path('/kaggle/input')
    if kaggle_in.exists():
        for d in sorted(kaggle_in.iterdir()):
            if not d.is_dir(): continue
            if (d / 'train').exists(): return d
            for sub in sorted(d.iterdir()):
                if sub.is_dir() and (sub / 'train').exists(): return sub
    for root in [Path('.'), Path('/content')]:
        if not root.exists(): continue
        if (root / 'train').exists(): return root
        for d in sorted(root.iterdir()):
            if d.is_dir() and not d.name.startswith('.') and (d / 'train').exists():
                return d
    return Path('.')

np.random.seed(42)
t0 = time.time()
DATA = find_data_root()
OUT  = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
OUT.mkdir(parents=True, exist_ok=True)

try:
    import google.colab; env = 'Google Colab'
except ImportError:
    env = 'Kaggle' if Path('/kaggle/input').exists() else 'Local'

print(f'Environment  : {env}')
print(f'Dataset path : {DATA.resolve()}')
print(f'train/ found : {(DATA / "train").exists()}')
print(f'test/  found : {(DATA / "test").exists()}')
print(f'LightGBM ver : {lgb.__version__}')

if not (DATA / 'train').exists():
    raise FileNotFoundError(
        f'Folder train/ tidak ditemukan di {DATA.resolve()}\n'
        'Solusi: isi variabel DATA_PATH di atas, atau pastikan zip sudah terupload.'
    )


Extracting datathon-task-1.zip...
Environment  : Google Colab
Dataset path : /content/dataset-task1
train/ found : True
test/  found : True
LightGBM ver : 4.6.0


In [2]:
# ── Load Data ──────────────────────────────────────────────────────────────────
print('Loading...')
speed_m1 = np.load(DATA / 'train' / 'train_speed_m1_1_11160.npy')
speed_m2 = np.load(DATA / 'train' / 'train_speed_m2_1_5039.npy')
test_X   = np.load(DATA / 'test'  / 'test_X_hist.npy').astype(np.float32)  # (540,15,1260)
train    = np.vstack([speed_m1, speed_m2]).astype(np.float32)               # (16199,1260)
matrix   = np.load(DATA / 'static' / 'matrix.npy').astype(np.float32)

with open(DATA / 'train' / 'train_text_m1_1_11160.json') as f: text_m1 = json.load(f)
with open(DATA / 'train' / 'train_text_m2_1_5039.json')  as f: text_m2 = json.load(f)
with open(DATA / 'test'  / 'test_texts.json')             as f: test_texts = json.load(f)

T, R = train.shape; H, F, H_n = 15, 3, 5
N_TEST = test_X.shape[0]
print(f'train:{train.shape}  test_X:{test_X.shape} | {time.time()-t0:.1f}s')


Loading...
train:(16199, 1260)  test_X:(540, 15, 1260) | 4.5s


In [3]:
# ── Spatial: neighbour average speed ──────────────────────────────────────────
deg      = matrix.sum(axis=1, keepdims=True)
adj_norm = (matrix / np.where(deg == 0, 1, deg)).astype(np.float32)
neigh      = train  @ adj_norm.T   # (T, R)
neigh_test = test_X @ adj_norm.T   # (540, 15, R)
print(f'Neighbour avg | {time.time()-t0:.1f}s')


Neighbour avg | 10.9s


In [4]:
# ── Text Features ──────────────────────────────────────────────────────────────
KEYWORDS = ['road closure', 'construction', 'accident', 'prohibit', 'announcement']
N_TEXT   = len(KEYWORDS) + 1

def text_feats(text: str) -> np.ndarray:
    t = text.lower()
    return np.array([float(kw in t) for kw in KEYWORDS] + [float(t.count('.'))],
                   dtype=np.float32)

train_tf = np.zeros((T, N_TEXT), dtype=np.float32)
for i in range(speed_m1.shape[0]):
    k = f'm1_{i+1}'
    if k in text_m1: train_tf[i] = text_feats(text_m1[k])
for i in range(speed_m2.shape[0]):
    k = f'm2_{i+1}'
    if k in text_m2: train_tf[speed_m1.shape[0]+i] = text_feats(text_m2[k])

test_tf = np.zeros((N_TEST, N_TEXT), dtype=np.float32)
for i in range(N_TEST):
    k = f'test_{i:05d}'
    if k in test_texts: test_tf[i] = text_feats(test_texts[k])

print(f'Text features: {N_TEXT} dims | {time.time()-t0:.1f}s')


Text features: 6 dims | 11.1s


In [5]:
# ── Build Training Dataset (Vectorized — no Python loop) ───────────────────────
STEP  = 2
n_win = T - H - F + 1
t_idx = np.arange(0, n_win, STEP)  # sampled window start indices
N_WIN = len(t_idx)

print(f'Windows: {N_WIN} (every {STEP}nd of {n_win}) => ~{N_WIN*R/1e6:.1f}M rows')

# Feature names (31 total)
feat_names = (
    [f'own_lag_{i+1}'   for i in range(H)] +     # 15
    [f'neigh_lag_{i+1}' for i in range(H_n)] +   # 5
    ['roll_mean5', 'roll_std15', 'roll_slope', 'neigh_mean5'] +  # 4 rolling
    [f'text_{kw.replace(" ","_")}' for kw in KEYWORDS] +
    ['text_n_incidents'] +                        # 6 text
    ['road_id']                                   # 1
)  # total = 31

def build_train_matrix_fast(f_idx):
    """Fully vectorized — no Python loop over windows."""
    # Own lags: train[t_idx[i] : t_idx[i]+H] for each window → use fancy indexing
    lag_idx  = t_idx[:, None] + np.arange(H)          # (N_WIN, H)
    own      = train[lag_idx]                          # (N_WIN, H, R)
    own_flat = own.transpose(0, 2, 1).reshape(-1, H)  # (N_WIN*R, H)

    # Neighbour lags: last H_n of the 15-step window
    nb_idx   = t_idx[:, None] + np.arange(H - H_n, H) # (N_WIN, H_n)
    nb       = neigh[nb_idx]                           # (N_WIN, H_n, R)
    nb_flat  = nb.transpose(0, 2, 1).reshape(-1, H_n) # (N_WIN*R, H_n)

    # Rolling features (computed on own_flat)
    roll_mean5  = own_flat[:, -5:].mean(1, keepdims=True)          # recent mean
    roll_std15  = own_flat.std(1, keepdims=True)                    # variability
    roll_slope  = (own_flat[:, -1:] - own_flat[:, :1]) / (H - 1)  # linear trend
    neigh_mean5 = nb_flat[:, -5:].mean(1, keepdims=True)           # recent neigh mean

    # Text: one row per window, repeat R times
    txt_idx = t_idx + H                             # (N_WIN,) — text at target step
    txt     = np.repeat(train_tf[txt_idx], R, axis=0)  # (N_WIN*R, N_TEXT)

    # Road IDs: tile [0..R-1] for each window
    rids = np.tile(np.arange(R, dtype=np.float32), N_WIN).reshape(-1, 1)  # (N_WIN*R, 1)

    # Targets
    tgt_idx = t_idx + H + f_idx                    # (N_WIN,)
    y = train[tgt_idx].reshape(-1)                 # (N_WIN*R,)

    X = np.hstack([own_flat, nb_flat, roll_mean5, roll_std15,
                   roll_slope, neigh_mean5, txt, rids]).astype(np.float32)
    return X, y.astype(np.float32)

print('Building train matrix...')
X0, y0 = build_train_matrix_fast(0)
print(f'  X:{X0.shape}  feat:{len(feat_names)} | {time.time()-t0:.1f}s')


Windows: 8091 (every 2nd of 16182) => ~10.2M rows
Building train matrix...
  X:(10194660, 31)  feat:31 | 16.7s


In [6]:
# ── Train LightGBM (one model per horizon) ─────────────────────────────────────
lgb_params = dict(
    objective         = 'regression_l1',  # MAE loss
    n_estimators      = 1000,
    learning_rate     = 0.03,
    num_leaves        = 255,
    min_child_samples = 100,
    feature_fraction  = 0.8,
    bagging_fraction  = 0.8,
    bagging_freq      = 5,
    lambda_l2         = 0.1,
    n_jobs            = -1,
    random_state      = 42,
    verbose           = -1,
)

models = {}
cat_feats = [feat_names.index('road_id')]

for fi, (h, f_idx) in enumerate(zip([5, 10, 15], [0, 1, 2])):
    print(f'Training h{h} model...')
    X, y = (X0, y0) if fi == 0 else build_train_matrix_fast(f_idx)
    X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.1, random_state=42)
    model = lgb.LGBMRegressor(**lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set            = [(X_val, y_val)],
        feature_name        = feat_names,
        callbacks           = [lgb.early_stopping(100, verbose=False),
                               lgb.log_evaluation(100)],
        categorical_feature = cat_feats,
    )
    models[h] = model
    val_mae = np.abs(model.predict(X_val) - y_val).mean()
    print(f'  h{h}: val MAE={val_mae:.4f} km/h | best_iter={model.best_iteration_} | {time.time()-t0:.1f}s')


Training h5 model...
[100]	valid_0's l1: 3.30935
[200]	valid_0's l1: 2.97676
[300]	valid_0's l1: 2.95282
[400]	valid_0's l1: 2.94376
[500]	valid_0's l1: 2.93896
[600]	valid_0's l1: 2.93526
[700]	valid_0's l1: 2.933
[800]	valid_0's l1: 2.93166
[900]	valid_0's l1: 2.93037
[1000]	valid_0's l1: 2.92944


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  h5: val MAE=2.9294 km/h | best_iter=1000 | 5224.7s
Training h10 model...
[100]	valid_0's l1: 3.55448
[200]	valid_0's l1: 3.23566
[300]	valid_0's l1: 3.20827
[400]	valid_0's l1: 3.19779
[500]	valid_0's l1: 3.19188
[600]	valid_0's l1: 3.18758
[700]	valid_0's l1: 3.18444
[800]	valid_0's l1: 3.18227
[900]	valid_0's l1: 3.18057
[1000]	valid_0's l1: 3.17938


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  h10: val MAE=3.1794 km/h | best_iter=1000 | 10195.1s
Training h15 model...
[100]	valid_0's l1: 3.67778
[200]	valid_0's l1: 3.36048
[300]	valid_0's l1: 3.33073
[400]	valid_0's l1: 3.31851
[500]	valid_0's l1: 3.31169
[600]	valid_0's l1: 3.30737
[700]	valid_0's l1: 3.30445
[800]	valid_0's l1: 3.30221
[900]	valid_0's l1: 3.30051
[1000]	valid_0's l1: 3.29926


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  h15: val MAE=3.2993 km/h | best_iter=1000 | 15429.9s


In [8]:
# ── Build Test Features & Predict (Vectorized) ────────────────────────────────
print('Predicting...')

# test_X: (540, 15, R), neigh_test: (540, 15, R)
own_t   = test_X.transpose(0, 2, 1)               # (540, R, H)
nb_t    = neigh_test[:, -H_n:, :].transpose(0,2,1) # (540, R, H_n)

roll_mean5_t  = own_t[:, :, -5:].mean(2, keepdims=True)          # (540, R, 1)
roll_std15_t  = own_t.std(2, keepdims=True)                       # (540, R, 1)
roll_slope_t  = (own_t[:, :, -1:] - own_t[:, :, :1]) / (H - 1)  # (540, R, 1)
neigh_mean5_t = nb_t[:, :, -5:].mean(2, keepdims=True)            # (540, R, 1)

# text: (540, N_TEXT) → (540, R, N_TEXT)
txt_t = np.repeat(test_tf[:, None, :], R, axis=1)                 # (540, R, N_TEXT)

# road ids: (540, R, 1)
rids_t = np.tile(np.arange(R, dtype=np.float32)[None, :, None], (N_TEST, 1, 1))

# Full test feature matrix: (540, R, 31) → reshape to (540*R, 31) per horizon
test_feat = np.concatenate(
    [own_t, nb_t, roll_mean5_t, roll_std15_t, roll_slope_t, neigh_mean5_t, txt_t, rids_t],
    axis=2
).astype(np.float32)  # (540, R, 31)

ids, spd = [], []
for fi, h in enumerate([5, 10, 15]):
    X_te = test_feat.reshape(-1, len(feat_names))  # (540*R, 31)
    preds = np.clip(models[h].predict(X_te), 0.0, 200.0).reshape(N_TEST, R)  # (540, R)
    for si in range(N_TEST):
        s = f'test_{si:05d}'
        for ri in range(R):
            ids.append(f'{s}_h{h}_r{ri}')
            spd.append(float(preds[si, ri]))

submission = pd.DataFrame({'id': ids, 'speed': spd})
sub_path = OUT / 'submission.csv'
submission.to_csv(sub_path, index=False)
print(f'Submission saved → {sub_path}')
print(f'Rows: {len(submission):,}')
print(submission['speed'].describe().round(3))
print(f'Total time: {time.time()-t0:.1f}s')


Predicting...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Submission saved → submission.csv
Rows: 2,041,200
count    2041200.000
mean          52.839
std           19.863
min            0.000
25%           42.757
50%           54.581
75%           67.490
max           97.055
Name: speed, dtype: float64
Total time: 17060.6s


## Summary

**LightGBM v2** — improvements over v1:

| | v1 | v2 |
|---|---|---|
| Training data | ~4M rows (STEP=5) | ~10M rows (STEP=2) |
| Trees | 500 | 1000 + early stopping |
| num_leaves | 127 | 255 |
| learning_rate | 0.05 | 0.03 |
| Features | 27 | 31 (+rolling stats) |
| Data build | Python loop | Numpy vectorized |
